### Carregar os dados

In [0]:
from pyspark.sql import functions as F
import re
import unicodedata

catalog = "first_catalog"
schema = "default"

landing_path = "/Volumes/first_catalog/default/logistica_vol/"

print("Landing:", landing_path)
print("Destino das tabelas:", f"{catalog}.{schema}")

Landing: /Volumes/first_catalog/default/logistica_vol/
Destino das tabelas: first_catalog.default


Visualizar as 5 primeiras linhas nas tabelas

In [0]:

ficheiros = {
    "clientes": "Clientes.csv",
    "pedidos": "Pedidos.csv",
    "produtos": "Produtos.csv",
    "vendas": "Vendas.csv"
}

dados_raw = {}

for nome, ficheiro in ficheiros.items():
    caminho = f"{landing_path.rstrip('/')}/{ficheiro}"

    print(f"\nA ler: {caminho}")

    df = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .option("inferSchema", "false")
        .csv(caminho)
    )

    dados_raw[nome] = df

    print(f"\n--- {ficheiro} ---")
    print("Número de colunas:", len(df.columns))
    print("Nomes das colunas:", df.columns)

    display(df.limit(5))


A ler: /Volumes/first_catalog/default/logistica_vol/Clientes.csv

--- Clientes.csv ---
Número de colunas: 8
Nomes das colunas: ['ID Cliente', 'Nome Cliente', 'Segmento', 'Cidade', 'Estado', 'Pais', 'Mercado', 'Regiao']


ID Cliente,Nome Cliente,Segmento,Cidade,Estado,Pais,Mercado,Regiao
RH-19495,Rick Hansen,Consumidor,New York City,New York,United States,US,East
JR-16210,Justin Ritter,Corporativo,Wollongong,New South Wales,Australia,APAC,Oceania
CR-12730,Craig Reiter,Consumidor,Brisbane,Queensland,Australia,APAC,Oceania
KM-16375,Katherine Murray,Outros,Berlin,Berlin,Germany,EU,Central
RH-9495,Rick Hansen,Consumidor,Dakar,Dakar,Senegal,Africa,Africa



A ler: /Volumes/first_catalog/default/logistica_vol/Pedidos.csv

--- Pedidos.csv ---
Número de colunas: 5
Nomes das colunas: ['ID Pedido', 'Data Pedido', 'Data Envio', 'Modo Envio', 'Prioridade Pedido']


ID Pedido,Data Pedido,Data Envio,Modo Envio,Prioridade Pedido
CA-2012-124891,31-07-2012,31-07-2012,Mesmo Dia,Critico
CA-2012-124891,31-07-2012,31-07-2012,Mesmo Dia,Critico
IN-2013-77878,05-02-2013,07-02-2013,Segunda Classe,Critico
IN-2013-71249,17-10-2013,18-10-2013,Primeira Classe,Medio
ES-2013-1579342,28-01-2013,30-01-2013,Primeira Classe,Medio



A ler: /Volumes/first_catalog/default/logistica_vol/Produtos.csv

--- Produtos.csv ---
Número de colunas: 4
Nomes das colunas: ['ID Produto', 'Categoria', 'SubCategoria', 'Nome Produto']


ID Produto,Categoria,SubCategoria,Nome Produto
TEC-AC-10003033,Tecnologia,Accessories,Plantronics CS510 - Over-the-Head monaural Wireless Headset System
TEC-AC-10003033,Tecnologia,Accessories,Plantronics CS510 - Over-the-Head monaural Wireless Headset System
FUR-CH-10003950,Moveis,Chairs,"Novimex Executive Leather Armchair, Black"
TEC-PH-10004664,Tecnologia,Phones,"Nokia Smart Phone, with Caller ID"
TEC-PH-10004583,Tecnologia,Phones,"Motorola Smart Phone, Cordless"



A ler: /Volumes/first_catalog/default/logistica_vol/Vendas.csv

--- Vendas.csv ---
Número de colunas: 6
Nomes das colunas: ['Pedido', 'Cliente', 'Produto', 'Valor Venda', 'Quantidade Vendida', 'Custo Envio']


Pedido,Cliente,Produto,Valor Venda,Quantidade Vendida,Custo Envio
CA-2012-124891,RH-19495,TEC-AC-10003033,"2309,65",7,"933,57"
IN-2013-77878,JR-16210,FUR-CH-10003950,"3709,395",9,"923,63"
IN-2013-71249,CR-12730,TEC-PH-10004664,"5175,171",9,"915,49"
ES-2013-1579342,KM-16375,TEC-PH-10004583,"2892,51",5,"910,16"
SG-2013-4320,RH-9495,TEC-SHA-10000501,"2832,96",8,"903,04"


### Normalizar os dados

In [0]:
def normalizar_nome_coluna(nome):
    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ascii", "ignore").decode("ascii")
    nome = nome.lower().strip()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    nome = nome.strip("_")

    return nome


for nome, df in dados_raw.items():
    novos_nomes = [
        normalizar_nome_coluna(c)
        for c in df.columns
    ]

    if len(novos_nomes) != len(set(novos_nomes)):
        raise ValueError(
            f"Existem nomes de colunas duplicados após normalização em {nome}"
        )

    for antigo, novo in zip(df.columns, novos_nomes):
        if antigo != novo:
            df = df.withColumnRenamed(antigo, novo)

    dados_raw[nome] = df

    print(f"\n{nome.upper()}")
    print(df.columns)





CLIENTES
['id_cliente', 'nome_cliente', 'segmento', 'cidade', 'estado', 'pais', 'mercado', 'regiao']

PEDIDOS
['id_pedido', 'data_pedido', 'data_envio', 'modo_envio', 'prioridade_pedido']

PRODUTOS
['id_produto', 'categoria', 'subcategoria', 'nome_produto']

VENDAS
['pedido', 'cliente', 'produto', 'valor_venda', 'quantidade_vendida', 'custo_envio']


### Criar as quatro tabelas Broze

In [0]:
batch_id = spark.sql("""
    SELECT date_format(
        current_timestamp(),
        'yyyyMMdd_HHmmss'
    ) AS batch_id
""").first()["batch_id"]

for nome, ficheiro in ficheiros.items():

    df_bronze = (
        dados_raw[nome]
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file",
            F.lit(ficheiro)
        )
        .withColumn(
            "_batch_id",
            F.lit(batch_id)
        )
    )

    tabela = f"{catalog}.{schema}.bronze_{nome}"

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela)
    )

    print(f"Tabela criada: {tabela}")



Tabela criada: first_catalog.default.bronze_clientes
Tabela criada: first_catalog.default.bronze_pedidos
Tabela criada: first_catalog.default.bronze_produtos
Tabela criada: first_catalog.default.bronze_vendas


Validar se os dados foram gravados (Python)

In [0]:
for nome in ficheiros:
    tabela = f"{catalog}.{schema}.bronze_{nome}"
    df = spark.table(tabela)

    print(f"\n--- {tabela} ---")
    print("Linhas:", df.count())
    print("Colunas:", len(df.columns))
    df.printSchema()
    display(df.limit(5))


--- first_catalog.default.bronze_clientes ---
Linhas: 1590
Colunas: 11
root
 |-- id_cliente: string (nullable = true)
 |-- nome_cliente: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- mercado: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



id_cliente,nome_cliente,segmento,cidade,estado,pais,mercado,regiao,_ingestion_timestamp,_source_file,_batch_id
RH-19495,Rick Hansen,Consumidor,New York City,New York,United States,US,East,2026-10-09T17:38:33.483Z,Clientes.csv,20261009_173827
JR-16210,Justin Ritter,Corporativo,Wollongong,New South Wales,Australia,APAC,Oceania,2026-10-09T17:38:33.483Z,Clientes.csv,20261009_173827
CR-12730,Craig Reiter,Consumidor,Brisbane,Queensland,Australia,APAC,Oceania,2026-10-09T17:38:33.483Z,Clientes.csv,20261009_173827
KM-16375,Katherine Murray,Outros,Berlin,Berlin,Germany,EU,Central,2026-10-09T17:38:33.483Z,Clientes.csv,20261009_173827
RH-9495,Rick Hansen,Consumidor,Dakar,Dakar,Senegal,Africa,Africa,2026-10-09T17:38:33.483Z,Clientes.csv,20261009_173827



--- first_catalog.default.bronze_pedidos ---
Linhas: 25036
Colunas: 8
root
 |-- id_pedido: string (nullable = true)
 |-- data_pedido: string (nullable = true)
 |-- data_envio: string (nullable = true)
 |-- modo_envio: string (nullable = true)
 |-- prioridade_pedido: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



id_pedido,data_pedido,data_envio,modo_envio,prioridade_pedido,_ingestion_timestamp,_source_file,_batch_id
CA-2012-124891,31-07-2012,31-07-2012,Mesmo Dia,Critico,2026-10-09T17:38:40.066Z,Pedidos.csv,20261009_173827
CA-2012-124891,31-07-2012,31-07-2012,Mesmo Dia,Critico,2026-10-09T17:38:40.066Z,Pedidos.csv,20261009_173827
IN-2013-77878,05-02-2013,07-02-2013,Segunda Classe,Critico,2026-10-09T17:38:40.066Z,Pedidos.csv,20261009_173827
IN-2013-71249,17-10-2013,18-10-2013,Primeira Classe,Medio,2026-10-09T17:38:40.066Z,Pedidos.csv,20261009_173827
ES-2013-1579342,28-01-2013,30-01-2013,Primeira Classe,Medio,2026-10-09T17:38:40.066Z,Pedidos.csv,20261009_173827



--- first_catalog.default.bronze_produtos ---
Linhas: 10293
Colunas: 7
root
 |-- id_produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- subcategoria: string (nullable = true)
 |-- nome_produto: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



id_produto,categoria,subcategoria,nome_produto,_ingestion_timestamp,_source_file,_batch_id
TEC-AC-10003033,Tecnologia,Accessories,Plantronics CS510 - Over-the-Head monaural Wireless Headset System,2026-10-09T17:38:43.642Z,Produtos.csv,20261009_173827
TEC-AC-10003033,Tecnologia,Accessories,Plantronics CS510 - Over-the-Head monaural Wireless Headset System,2026-10-09T17:38:43.642Z,Produtos.csv,20261009_173827
FUR-CH-10003950,Moveis,Chairs,"Novimex Executive Leather Armchair, Black",2026-10-09T17:38:43.642Z,Produtos.csv,20261009_173827
TEC-PH-10004664,Tecnologia,Phones,"Nokia Smart Phone, with Caller ID",2026-10-09T17:38:43.642Z,Produtos.csv,20261009_173827
TEC-PH-10004583,Tecnologia,Phones,"Motorola Smart Phone, Cordless",2026-10-09T17:38:43.642Z,Produtos.csv,20261009_173827



--- first_catalog.default.bronze_vendas ---
Linhas: 51290
Colunas: 9
root
 |-- pedido: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- valor_venda: string (nullable = true)
 |-- quantidade_vendida: string (nullable = true)
 |-- custo_envio: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



pedido,cliente,produto,valor_venda,quantidade_vendida,custo_envio,_ingestion_timestamp,_source_file,_batch_id
CA-2012-124891,RH-19495,TEC-AC-10003033,"2309,65",7,"933,57",2026-10-09T17:38:47.471Z,Vendas.csv,20261009_173827
IN-2013-77878,JR-16210,FUR-CH-10003950,"3709,395",9,"923,63",2026-10-09T17:38:47.471Z,Vendas.csv,20261009_173827
IN-2013-71249,CR-12730,TEC-PH-10004664,"5175,171",9,"915,49",2026-10-09T17:38:47.471Z,Vendas.csv,20261009_173827
ES-2013-1579342,KM-16375,TEC-PH-10004583,"2892,51",5,"910,16",2026-10-09T17:38:47.471Z,Vendas.csv,20261009_173827
SG-2013-4320,RH-9495,TEC-SHA-10000501,"2832,96",8,"903,04",2026-10-09T17:38:47.471Z,Vendas.csv,20261009_173827


Exercicio A - Verificar valores nulos e vazios

In [0]:
from pyspark.sql import functions as F

tabelas = {
    "clientes": "first_catalog.default.bronze_clientes",
    "pedidos": "first_catalog.default.bronze_pedidos",
    "produtos": "first_catalog.default.bronze_produtos",
    "vendas": "first_catalog.default.bronze_vendas"
}

for nome, tabela in tabelas.items():
    df = spark.table(tabela)

    colunas_negocio = [
        c for c in df.columns
        if not c.startswith("_")
    ]

    expressoes = [
        F.sum(
            F.when(
                F.col(c).isNull() |
                (F.trim(F.col(c)) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in colunas_negocio
    ]

    perfil = df.agg(*expressoes)

    print(f"\n--- Qualidade: {nome.upper()} ---")
    display(perfil)


--- Qualidade: CLIENTES ---


id_cliente,nome_cliente,segmento,cidade,estado,pais,mercado,regiao
0,0,0,0,0,0,0,0



--- Qualidade: PEDIDOS ---


id_pedido,data_pedido,data_envio,modo_envio,prioridade_pedido
0,0,0,0,0



--- Qualidade: PRODUTOS ---


id_produto,categoria,subcategoria,nome_produto
0,0,0,0



--- Qualidade: VENDAS ---


pedido,cliente,produto,valor_venda,quantidade_vendida,custo_envio
0,0,0,0,0,0


Exercicios - Encontrar valores duplicados

In [0]:
for nome, tabela in tabelas.items():
    df = spark.table(tabela)

    colunas_negocio = [
        c for c in df.columns
        if not c.startswith("_")
    ]

    total = df.count()
    distintos = df.select(*colunas_negocio).dropDuplicates().count()

    print(f"\n--- Duplicados: {nome.upper()} ---")
    print("Total de linhas:", total)
    print("Linhas distintas:", distintos)
    print("Repetições exatas:", total - distintos)


--- Duplicados: CLIENTES ---
Total de linhas: 1590
Linhas distintas: 1590
Repetições exatas: 0

--- Duplicados: PEDIDOS ---
Total de linhas: 25036
Linhas distintas: 25035
Repetições exatas: 1

--- Duplicados: PRODUTOS ---
Total de linhas: 10293
Linhas distintas: 10292
Repetições exatas: 1

--- Duplicados: VENDAS ---
Total de linhas: 51290
Linhas distintas: 51290
Repetições exatas: 0


Exercicio C - Investigar as chaves

In [0]:
chaves = {
    "clientes": ("first_catalog.default.bronze_clientes", "id_cliente"),
    "pedidos": ("first_catalog.default.bronze_pedidos", "id_pedido"),
    "produtos": ("first_catalog.default.bronze_produtos", "id_produto"),
    "vendas": ("first_catalog.default.bronze_vendas", "pedido")
}

for nome, (tabela, coluna) in chaves.items():
    df = spark.table(tabela)

    print(f"\n--- Chave candidata: {nome}.{coluna} ---")

    (
        df.groupBy(coluna)
        .agg(F.count("*").alias("ocorrencias"))
        .filter(
            F.col(coluna).isNull() |
            (F.trim(F.col(coluna)) == "") |
            (F.col("ocorrencias") > 1)
        )
        .orderBy(F.desc("ocorrencias"))
        .show(20, truncate=False)
    )


--- Chave candidata: clientes.id_cliente ---
+----------+-----------+
|id_cliente|ocorrencias|
+----------+-----------+
+----------+-----------+


--- Chave candidata: pedidos.id_pedido ---
+--------------+-----------+
|id_pedido     |ocorrencias|
+--------------+-----------+
|CA-2012-124891|2          |
+--------------+-----------+


--- Chave candidata: produtos.id_produto ---
+---------------+-----------+
|id_produto     |ocorrencias|
+---------------+-----------+
|TEC-AC-10003033|2          |
+---------------+-----------+


--- Chave candidata: vendas.pedido ---
+---------------+-----------+
|pedido         |ocorrencias|
+---------------+-----------+
|CA-2014-100111 |14         |
|NI-2014-8880   |13         |
|IN-2012-41261  |13         |
|TO-2014-9950   |13         |
|IN-2013-42311  |13         |
|MX-2014-166541 |13         |
|MX-2013-127705 |12         |
|IN-2011-76625  |12         |
|MX-2013-142678 |12         |
|CA-2014-157987 |12         |
|IN-2014-15263  |12         |
|ES-20

Exercicio D - Inspecionar datas e valores numéricos

In [0]:
print("DATAS DOS PEDIDOS")

display(
    spark.table("first_catalog.default.bronze_pedidos")
    .select("data_pedido", "data_envio")
    .distinct()
    .limit(20)
)

print("VALORES DE VENDAS")

display(
    spark.table("first_catalog.default.bronze_vendas")
    .select("valor_venda", "quantidade_vendida", "custo_envio")
    .distinct()
    .limit(30)
)

DATAS DOS PEDIDOS


data_pedido,data_envio
31-07-2012,31-07-2012
05-02-2013,07-02-2013
17-10-2013,18-10-2013
28-01-2013,30-01-2013
05-11-2013,06-11-2013
28-06-2013,01-07-2013
07-11-2011,09-11-2011
14-04-2012,18-04-2012
14-10-2014,21-10-2014
28-01-2012,31-01-2012


VALORES DE VENDAS


valor_venda,quantidade_vendida,custo_envio
"2309,65",7,"933,57"
"3709,395",9,"923,63"
"5175,171",9,"915,49"
"2892,51",5,"910,16"
"2832,96",8,"903,04"
"2862,675",5,"897,35"
"1822,08",4,"894,77"
"5244,84",6,"878,38"
"5083,96",5,"867,69"
"4297,644",13,"865,74"


Exercicio E - COmpletar o perfil de qualidade das 4 tabelas 

In [0]:

catalog = "first_catalog"
schema = "default"

tabelas = {
    "clientes": f"{catalog}.{schema}.bronze_clientes",
    "pedidos": f"{catalog}.{schema}.bronze_pedidos",
    "produtos": f"{catalog}.{schema}.bronze_produtos",
    "vendas": f"{catalog}.{schema}.bronze_vendas"
}

print("Tabelas configuradas:")
for nome, tabela in tabelas.items():
    print(f"{nome}: {tabela}")

Tabelas configuradas:
clientes: first_catalog.default.bronze_clientes
pedidos: first_catalog.default.bronze_pedidos
produtos: first_catalog.default.bronze_produtos
vendas: first_catalog.default.bronze_vendas


In [0]:


from pyspark.sql import functions as F

for nome, tabela in tabelas.items():
    df = spark.table(tabela)

    colunas = [c for c in df.columns if not c.startswith("_")]

    if not colunas:
        print(f"A tabela {nome.upper()} não tem colunas para analisar.")
        continue

    perfil = df.agg(*[
        F.sum(
            F.when(
                F.col(c).isNull() |
                (F.trim(F.col(c).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in colunas
    ])

    print(f"Qualidade de {nome.upper()}")
    display(perfil)

Qualidade de CLIENTES


id_cliente,nome_cliente,segmento,cidade,estado,pais,mercado,regiao
0,0,0,0,0,0,0,0


Qualidade de PEDIDOS


id_pedido,data_pedido,data_envio,modo_envio,prioridade_pedido
0,0,0,0,0


Qualidade de PRODUTOS


id_produto,categoria,subcategoria,nome_produto
0,0,0,0


Qualidade de VENDAS


pedido,cliente,produto,valor_venda,quantidade_vendida,custo_envio
0,0,0,0,0,0


Exercicio F - Investigar os dois duplicados

In [0]:
pedidos = spark.table("first_catalog.default.bronze_pedidos")

display(
    pedidos
    .filter(F.col("id_pedido") == "CA-2012-124891")
    .orderBy("data_pedido", "data_envio")
)

id_pedido,data_pedido,data_envio,modo_envio,prioridade_pedido,_ingestion_timestamp,_source_file,_batch_id
CA-2012-124891,31-07-2012,31-07-2012,Mesmo Dia,Critico,2026-10-09T17:38:40.066Z,Pedidos.csv,20261009_173827
CA-2012-124891,31-07-2012,31-07-2012,Mesmo Dia,Critico,2026-10-09T17:38:40.066Z,Pedidos.csv,20261009_173827


In [0]:
produtos = spark.table("first_catalog.default.bronze_produtos")

display(
    produtos
    .filter(F.col("id_produto") == "TEC-AC-10003033")
)

id_produto,categoria,subcategoria,nome_produto,_ingestion_timestamp,_source_file,_batch_id
TEC-AC-10003033,Tecnologia,Accessories,Plantronics CS510 - Over-the-Head monaural Wireless Headset System,2026-10-09T17:38:43.642Z,Produtos.csv,20261009_173827
TEC-AC-10003033,Tecnologia,Accessories,Plantronics CS510 - Over-the-Head monaural Wireless Headset System,2026-10-09T17:38:43.642Z,Produtos.csv,20261009_173827


Exericio G - Validar as datas e os valores numéricos 

In [0]:
pedidos = spark.table("first_catalog.default.bronze_pedidos")

pedidos_datas = (
    pedidos
    .withColumn(
        "data_pedido_convertida",
        F.to_date("data_pedido", "dd-MM-yyyy")
    )
    .withColumn(
        "data_envio_convertida",
        F.to_date("data_envio", "dd-MM-yyyy")
    )
)

display(
    pedidos_datas.select(
        "data_pedido",
        "data_pedido_convertida",
        "data_envio",
        "data_envio_convertida"
    ).limit(20)
)

print(
    "Datas de pedido inválidas:",
    pedidos_datas.filter(
        F.col("data_pedido_convertida").isNull()
    ).count()
)

print(
    "Datas de envio inválidas:",
    pedidos_datas.filter(
        F.col("data_envio_convertida").isNull()
    ).count()
)

data_pedido,data_pedido_convertida,data_envio,data_envio_convertida
31-07-2012,2012-07-31,31-07-2012,2012-07-31
31-07-2012,2012-07-31,31-07-2012,2012-07-31
05-02-2013,2013-02-05,07-02-2013,2013-02-07
17-10-2013,2013-10-17,18-10-2013,2013-10-18
28-01-2013,2013-01-28,30-01-2013,2013-01-30
05-11-2013,2013-11-05,06-11-2013,2013-11-06
28-06-2013,2013-06-28,01-07-2013,2013-07-01
07-11-2011,2011-11-07,09-11-2011,2011-11-09
14-04-2012,2012-04-14,18-04-2012,2012-04-18
14-10-2014,2014-10-14,21-10-2014,2014-10-21


Datas de pedido inválidas: 0
Datas de envio inválidas: 0


Validar valores e venda

In [0]:
vendas = spark.table("first_catalog.default.bronze_vendas")

vendas_teste = (
    vendas
    .withColumn(
        "valor_venda_teste",
        F.regexp_replace(
            F.trim(F.col("valor_venda")),
            ",",
            "."
        ).cast("decimal(18,3)")
    )
    .withColumn(
        "quantidade_teste",
        F.trim(F.col("quantidade_vendida")).cast("decimal(18,3)")
    )
    .withColumn(
        "custo_envio_teste",
        F.regexp_replace(
            F.trim(F.col("custo_envio")),
            ",",
            "."
        ).cast("decimal(18,3)")
    )
)

display(
    vendas_teste.select(
        "valor_venda",
        "valor_venda_teste",
        "quantidade_vendida",
        "quantidade_teste",
        "custo_envio",
        "custo_envio_teste"
    ).limit(20)
)

valor_venda,valor_venda_teste,quantidade_vendida,quantidade_teste,custo_envio,custo_envio_teste
"2309,65",2309.650,7,7.000,"933,57",933.570
"3709,395",3709.395,9,9.000,"923,63",923.630
"5175,171",5175.171,9,9.000,"915,49",915.490
"2892,51",2892.510,5,5.000,"910,16",910.160
"2832,96",2832.960,8,8.000,"903,04",903.040
"2862,675",2862.675,5,5.000,"897,35",897.350
"1822,08",1822.080,4,4.000,"894,77",894.770
"5244,84",5244.840,6,6.000,"878,38",878.380
"5083,96",5083.960,5,5.000,"867,69",867.690
"4297,644",4297.644,13,13.000,"865,74",865.740


Detectando conversoes falhadas

In [0]:
print(
    "Valor de venda inválido:",
    vendas_teste.filter(
        F.col("valor_venda").isNotNull() &
        (F.trim(F.col("valor_venda")) != "") &
        F.col("valor_venda_teste").isNull()
    ).count()
)

print(
    "Quantidade inválida:",
    vendas_teste.filter(
        F.col("quantidade_vendida").isNotNull() &
        (F.trim(F.col("quantidade_vendida")) != "") &
        F.col("quantidade_teste").isNull()
    ).count()
)
 
print(
    "Custo de envio inválido:",
    vendas_teste.filter(
        F.col("custo_envio").isNotNull() &
        (F.trim(F.col("custo_envio")) != "") &
        F.col("custo_envio_teste").isNull()
    ).count()
)

Valor de venda inválido: 0
Quantidade inválida: 0
Custo de envio inválido: 0


Verificar as relações entre as quatro tabelas

In [0]:
from pyspark.sql import functions as F

clientes = spark.table("first_catalog.default.bronze_clientes")
pedidos = spark.table("first_catalog.default.bronze_pedidos")
produtos = spark.table("first_catalog.default.bronze_produtos")
vendas = spark.table("first_catalog.default.bronze_vendas")

# Obter os identificadores distintos das tabelas de referência
clientes_ref = clientes.select(
    F.trim("id_cliente").alias("cliente_id")
).distinct()

pedidos_ref = pedidos.select(
    F.trim("id_pedido").alias("pedido_id")
).distinct()

produtos_ref = produtos.select(
    F.trim("id_produto").alias("produto_id")
).distinct()

# Identificadores presentes nas vendas
vendas_cliente = vendas.select(
    F.trim("cliente").alias("cliente_id")
).distinct()

vendas_pedido = vendas.select(
    F.trim("pedido").alias("pedido_id")
).distinct()

vendas_produto = vendas.select(
    F.trim("produto").alias("produto_id")
).distinct()

# Identificar referências sem correspondência
clientes_em_falta = vendas_cliente.join(
    clientes_ref,
    vendas_cliente.cliente_id == clientes_ref.cliente_id,
    "left_anti"
)

pedidos_em_falta = vendas_pedido.join(
    pedidos_ref,
    vendas_pedido.pedido_id == pedidos_ref.pedido_id,
    "left_anti"
)

produtos_em_falta = vendas_produto.join(
    produtos_ref,
    vendas_produto.produto_id == produtos_ref.produto_id,
    "left_anti"
)

print("Clientes referenciados sem correspondência:",
      clientes_em_falta.count())

print("Pedidos referenciados sem correspondência:",
      pedidos_em_falta.count())

print("Produtos referenciados sem correspondência:",
      produtos_em_falta.count())

display(clientes_em_falta.limit(20))
display(pedidos_em_falta.limit(20))
display(produtos_em_falta.limit(20))

Clientes referenciados sem correspondência: 0
Pedidos referenciados sem correspondência: 0
Produtos referenciados sem correspondência: 0


cliente_id


pedido_id


produto_id
